In [100]:
#import the kaggle token to access the dataset
import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")


In [101]:
#Import the kaggle Dataset and unzip it
!kaggle datasets download -d mobeenfatimah/fake-news-detection-dataset-6000-news-articles
!unzip fake-news-detection-dataset-6000-news-articles.zip


Dataset URL: https://www.kaggle.com/datasets/mobeenfatimah/fake-news-detection-dataset-6000-news-articles
License(s): CC-BY-SA-4.0
fake-news-detection-dataset-6000-news-articles.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  fake-news-detection-dataset-6000-news-articles.zip
replace news_dataset.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [102]:
# import pandas and make it read the file
import pandas as pd

df = pd.read_csv('news_dataset.csv')

In [103]:
# Check for null values and remove

df.isnull().sum()
df.isnull().values.any()
df.info()
df.dropna()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    6000 non-null   object
 1   text     6000 non-null   object
 2   subject  6000 non-null   object
 3   date     6000 non-null   object
 4   label    6000 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 234.5+ KB


,title,text,subject,date,label
0,WATCH: Trump Just Told All The Anti-Gay Bigot...,A whole lot of evangelical Trump voters just d...,News,"November 13, 2016",0
1,"China backs U.N. call for justice in Yemen, U....",GENEVA (Reuters) - China signaled on Wednesday...,worldnews,"September 13, 2017",1
2,THE PEOPLE’S PRESIDENT: Trump Meets With Coal ...,,politics,"Feb 16, 2017",0
3,WSJ REPORTER RIPS INTO DEM CANDIDATES For Thei...,THE WSJ S MARY KISSEL NAILS IT ON THE DEM DEBA...,politics,"Nov 15, 2015",0
4,Lawmakers aim to delay U.S. ceding control of ...,WASHINGTON (Reuters) - Critics of a plan for t...,politicsNews,"September 13, 2016",1
...,...,...,...,...,...
5995,Exiled Venezuelan opposition magistrates resur...,SANTIAGO (Reuters) - A group of opposition-app...,worldnews,"October 19, 2017",1
5996,Plague outbreak in Madagascar kills 20: WHO,NAIROBI (Reuters) - An outbreak of plague has ...,worldnews,"September 29, 2017",1
5997,"Trump to visit Asia in November, North Korea i...",WASHINGTON (Reuters) - Donald Trump will trave...,worldnews,"September 29, 2017",1
5998,Melania Trump calls taped comments by Donald T...,(Reuters) - Melania Trump rose to her husband’...,politicsNews,"October 18, 2016",1


In [104]:
# clean the code to lowercase "title", "text", and "subject"
df['title'] = df['title'].str.lower()
df['text'] = df['text'].str.lower()
df['subject'] = df['subject'].str.lower()

In [105]:
# Remove the punctuation in title and text
df['title'] = df['title'].str.replace(r'[^a-zA-Z0-9\s]+', ' ', regex=True)
df['text'] = df['text'].str.replace(r'[^a-zA-Z0-9\s]+', ' ', regex=True)
df['subject'] = df['subject'].str.replace(r'[^a-zA-Z0-9\s]+', '' , regex=True)
print(df)

                                                  title  \
0      watch  trump just told all the anti gay bigot...   
1     china backs u n  call for justice in yemen  u ...   
2     the people s president  trump meets with coal ...   
3     wsj reporter rips into dem candidates for thei...   
4     lawmakers aim to delay u s  ceding control of ...   
...                                                 ...   
5995  exiled venezuelan opposition magistrates resur...   
5996        plague outbreak in madagascar kills 20  who   
5997  trump to visit asia in november  north korea i...   
5998  melania trump calls taped comments by donald t...   
5999   boycottpenzeys  hateful  divisive penzeys spi...   

                                                   text       subject  \
0     a whole lot of evangelical trump voters just d...          news   
1     geneva  reuters    china signaled on wednesday...     worldnews   
2                                                            politics   

In [106]:
# Drop duplicates and Remove the words stopwords from title and text
df = df.drop_duplicates(subset=['title', 'text', 'subject']).copy()
generalwords = ['a', 'an', 'is', 'the', 'to', 'in', 'she', 'it', 'they', 'in', 'on', 'at', 'with']
regexp = r'\b(' + '|'.join(generalwords) + r')\b'
df['title'] = df['title'].str.replace(regexp, ' ', regex = True)
df['text'] = df['text'].str.replace(
    r'^[a-z\s]+\(\s*reuters\s*\)\s*-\s*', '', regex=True
)



In [107]:
# Begin the lemmanization process using spacy
import spacy

nlp = spacy.load("en_core_web_sm")

def lemmanization(text):
  doc = nlp(text)
  return " ".join([token.lemma_ for token in doc])

In [108]:
# Tokenization using .split built in pandas
df['tokens'] = df['text'].str.split()

In [109]:
df['text_lemmatized'] = df['text'].apply(lemmanization)
display(df.head())

,title,text,subject,date,label,tokens,text_lemmatized
0,watch trump just told all anti gay bigots ...,a whole lot of evangelical trump voters just d...,news,"November 13, 2016",0,"[a, whole, lot, of, evangelical, trump, voters...",a whole lot of evangelical trump voter just di...
1,china backs u n call for justice yemen u s...,geneva reuters china signaled on wednesday...,worldnews,"September 13, 2017",1,"[geneva, reuters, china, signaled, on, wednesd...",geneva reuters china signal on wednesday...
2,people s president trump meets coal worke...,,politics,"Feb 16, 2017",0,[],
3,wsj reporter rips into dem candidates for thei...,the wsj s mary kissel nails it on the dem deba...,politics,"Nov 15, 2015",0,"[the, wsj, s, mary, kissel, nails, it, on, the...",the wsj s mary kissel nail it on the dem debat...
4,lawmakers aim delay u s ceding control of i...,washington reuters critics of a plan for t...,politicsnews,"September 13, 2016",1,"[washington, reuters, critics, of, a, plan, fo...",washington reuters critic of a plan for ...


In [110]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(df['text_lemmatized']).toarray()
y_label = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_label, test_size=0.2, random_state=42, stratify=y_label
)

classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)
print(f'TF-IDF Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))



TF-IDF Accuracy: 0.9808
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       600
           1       0.97      0.99      0.98       599

    accuracy                           0.98      1199
   macro avg       0.98      0.98      0.98      1199
weighted avg       0.98      0.98      0.98      1199



In [111]:
# Install necessary libraries for web scraping
!pip install requests beautifulsoup4

In [112]:
import requests
from bs4 import BeautifulSoup

def fetch_text_from_url(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        for element in soup(["script", "style", "header", "footer", "nav", "aside", "form"]):
            element.decompose()

        article_body = soup.find('div', class_=lambda c: c and 'Page-articleBody' in c)
        if not article_body:
            article_body = soup.find('article') or soup

        paragraphs = article_body.find_all('p')
        valid_paragraphs = [p.get_text().strip() for p in paragraphs if len(p.get_text().strip()) > 40]

        article_text = ' '.join(valid_paragraphs)

        return article_text.strip()

    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL {url}: {e}")
        return ""
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return ""

In [113]:
import re
import spacy

nlp = spacy.load('en_core_web_sm')


def preprocess_new_text(text):
  text = text.lower()

  text = re.sub(r'^[a-z\s,]+\(\s*[a-z]+\s*\)\s*[\-\u2014]\s*', '', text)
  text = re.sub(r'[^a-zA-Z\s]+', ' ', text)

  doc = nlp(text)

  tokens = [token.lemma_
      for token in doc
      if not token.is_stop and len(token.lemma_) > 2 ]

  return tokens

In [114]:
def predict_fake_news(url, tfidf_vectorizer, classifier_model):
  print(f'Processing URL: {url}')

  raw_text = fetch_text_from_url(url)
  if not raw_text:
    return 'Could not fetch content or content was empty.'
  print(f'Fetched content (first 200 chars): {raw_text[:200]}...')

  processed_tokens = preprocess_new_text(raw_text)
  if not processed_tokens:
    return 'No valid tokens after preprocessing.'
  print(f'Processed tokens (first 10): {processed_tokens[:10]}...')

  processed_text = ' '.join(processed_tokens)
  vectorized_text = tfidf_vectorizer.transform([processed_text]).toarray()

  prediction = classifier_model.predict(vectorized_text)
  probability = classifier_model.predict_proba(vectorized_text)

  result = 'Real News' if prediction[0] == 1 else 'Fake News'
  print(
      f'Prediction: {result} (Probability: Fake={probability[0][0]:.4f},'
      f' Real={probability[0][1]:.4f})'
  )
  return result

In [115]:
example_url = 'https://apnews.com/article/europe-wildfires-climate-bombs-world-war-5390dcf3c92c41ec44c2b0a4d9d80157'

if 'vectorizer' not in globals() or 'classifier' not in globals():
  print('Error: TF-IDF vectorizer or classifier not found.')
else:
  prediction_result = predict_fake_news(
      url=example_url,
      tfidf_vectorizer=vectorizer,
      classifier_model=classifier,
  )
  print(f'\nFinal Prediction for {example_url}: {prediction_result}')

Processing URL: https://apnews.com/article/europe-wildfires-climate-bombs-world-war-5390dcf3c92c41ec44c2b0a4d9d80157
Fetched content (first 200 chars): Members of a bomb disposal unit, Dieter Schwaetzler, left, and Rene Bennert sit next to 1.8 ton World War II bomb after defusing it in Frankfurt, Germany, Sept. 3, 2017. (AP Photo/Michael Probst, File...
Processed tokens (first 10): ['member', 'bomb', 'disposal', 'unit', 'dieter', 'schwaetzler', 'leave', 'rene', 'bennert', 'sit']...
Prediction: Real News (Probability: Fake=0.3960, Real=0.6040)

Final Prediction for https://apnews.com/article/europe-wildfires-climate-bombs-world-war-5390dcf3c92c41ec44c2b0a4d9d80157: Real News
